# 02. Political Text Preprocessing Pipeline

- **Role/Owner:** Member 2 (ML & Data Engineering Lead)
- **Core Responsibility:** Clean and normalize political claim text (HTML entity decoding, Unicode NFKD + homoglyph folding, emoji stripping, URL / @mention masking, camelCase hashtag splitting, sensational punctuation collapsing), then build the stratified train/val/test splits.
- **Key Interface/Contract:** Consumes the raw PolitiFact CSVs and PHEME cascade pickle; outputs unified train/val/test CSVs to `backend/data/processed/`. The cleaning functions are imported from `app/services/preprocessor.py`, the single copy shared with the FastAPI service.

In [1]:
# Environment & Dependency Setup
import re
import html
import time
import pickle
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)

print('02_preprocessing dependencies imported successfully.')

02_preprocessing dependencies imported successfully.


## Part 1: Raw Corpus Loading
We rebuild the unified raw corpus exactly as audited in `01_data_exploration.ipynb`: PolitiFact claim titles (432 fake, 624 real) plus the root/seed tweet of each of the 6,425 PHEME cascades. Only seed texts enter the NLP corpus; reply trees stay in `cascades` for graph features.

In [2]:
# Load PolitiFact claims
fake = pd.read_csv('../data/fakenewsnet_politifact/politifact_fake.csv')
real = pd.read_csv('../data/fakenewsnet_politifact/politifact_real.csv')

politifact_df = pd.concat([
    fake.assign(label=1),
    real.assign(label=0),
], ignore_index=True)
politifact_df = pd.DataFrame({
    'id': politifact_df['id'].astype(str),
    'text': politifact_df['title'],
    'label': politifact_df['label'],
    'source': 'politifact',
    'event': 'politifact',
})

# Load PHEME cascades and extract root/seed tweets
with open('../data/pheme_political/pheme_cascades.gpickle', 'rb') as f:
    cascades = pickle.load(f)

pheme_rows = []
for g in cascades:
    thread_id = str(g.graph.get('thread_id'))
    root = g.nodes[thread_id] if thread_id in g.nodes else {}
    pheme_rows.append({
        'id': thread_id,
        'text': str(root.get('text', '')),
        'label': int(g.graph.get('label')),
        'source': 'pheme',
        'event': str(g.graph.get('event')),
    })
pheme_df = pd.DataFrame(pheme_rows)

raw_df = pd.concat([politifact_df, pheme_df], ignore_index=True)
assert len(raw_df) == 7481, f'Expected 7,481 rows, found {len(raw_df)}'

print(f'Raw corpus: {len(raw_df):,} records | PolitiFact: {len(politifact_df):,} | PHEME: {len(pheme_df):,}')
print(f'Misinformation share: {raw_df["label"].mean():.1%}')

Raw corpus: 7,481 records | PolitiFact: 1,056 | PHEME: 6,425
Misinformation share: 37.9%


## Part 2: Cleaning Function Implementation
Each stage is a small pure function in `backend/app/services/preprocessor.py`. The notebook imports them, so the FastAPI service and these splits always use the same code; `backend/tests/test_preprocessor.py` locks in the behaviour. Stage order matters:

1. **HTML entities** (`&amp;` → `&`) — 322 PHEME tweets carry escaped entities.
2. **Unicode normalization** — folds common Cyrillic/Greek homoglyphs to Latin, applies NFKD (fullwidth letters, ligatures, `…` → `...`), then drops combining accents (`café` → `cafe`). NFKD alone does *not* map Cyrillic `а` to Latin `a`, hence the explicit homoglyph table.
3. **Emoji stripping** — emoji, pictographs, dingbats, flags, ZWJ and variation selectors.
4. **Entity masking** — URLs → `[URL]`, handles → `@USER`. Runs before hashtag splitting so `#` fragments inside URLs are never touched.
5. **Hashtag splitting** — `#StopTheSteal` → `Stop The Steal`, `#BBCNews` → `BBC News`, `#COVID19` → `COVID 19`; lowercase tags just lose the `#`.
6. **Punctuation** — runs of `!`/`?` collapse to one mark set off as its own token (`Fraud!!!!!` → `Fraud !`), so the sensationalism signal survives as a consistent token; `....` → `.`.
7. **Whitespace collapse and word truncation** at `max_length` words.

In [3]:
import sys

sys.path.insert(0, '..')  # make backend/ importable so the service module is the single source of truth

from app.services.preprocessor import (
    EMOJI_PATTERN,
    MENTION_PATTERN,
    URL_PATTERN,
    clean_political_text,
    mask_entities,
    normalize_punctuation,
    normalize_unicode,
    remove_emojis,
    split_hashtags,
)

print('Cleaning pipeline imported from app.services.preprocessor: normalize_unicode, remove_emojis, '
      'mask_entities, split_hashtags, normalize_punctuation, clean_political_text')

Cleaning pipeline imported from app.services.preprocessor: normalize_unicode, remove_emojis, mask_entities, split_hashtags, normalize_punctuation, clean_political_text


## Part 3: Edge-Case Unit Assertion Suite
Every stage is checked in isolation and through the full pipeline, including the empty, `None`, NaN and purely symbolic inputs that must return a string without raising.

In [4]:
cases = [
    # (input, expected, description)
    ('#StopTheSteal', 'Stop The Steal', 'CamelCase hashtag'),
    ('#FergusonDecision', 'Ferguson Decision', 'PascalCase hashtag'),
    ('#BBCNews live', 'BBC News live', 'acronym hashtag'),
    ('#COVID19', 'COVID 19', 'alphanumeric hashtag'),
    ('#ferguson', 'ferguson', 'lowercase hashtag'),
    ('##ParisShooting', 'Paris Shooting', 'doubled hash typo'),
    ('#black_lives_matter', 'black lives matter', 'underscore hashtag'),
    ('Breaking: https://t.co/xyz123', 'Breaking: [URL]', 'https URL'),
    ('see http://a.co/1 and www.b.com/x', 'see [URL] and [URL]', 'multiple URLs'),
    ('@realDonaldTrump says hello', '@USER says hello', 'mention'),
    ('RT .@rolandsmartin: wow', 'RT .@USER: wow', 'retweet mention'),
    ('mail me at a@b.com', 'mail me at a@b.com', 'email is not a mention'),
    ('Fraud!!!!!', 'Fraud !', 'repeated exclamation'),
    ('Really???', 'Really ?', 'repeated question'),
    ('What?!?!', 'What ?', 'mixed !? run'),
    ('Wait....', 'Wait.', 'repeated dots'),
    ('Wait…', 'Wait.', 'unicode ellipsis'),
    ('Price is 3.5%', 'Price is 3.5%', 'decimals untouched'),
    ('Cops &amp; protesters', 'Cops & protesters', 'HTML entity'),
    ('Hard to believe 😂😂😂', 'Hard to believe', 'emoji'),
    ('Flag 🇺🇸 here ❤️', 'Flag here', 'flag and heart emoji'),
    ('Тrump', 'Trump', 'Cyrillic T homoglyph'),
    ('Оbаmа', 'Obama', 'Cyrillic O/a homoglyphs'),
    ('café naïve', 'cafe naive', 'accents stripped'),
    ('ＦＡＫＥ ﬁle', 'FAKE file', 'fullwidth letters and ligature'),
    ('', '', 'empty string'),
    ('   \n\t ', '', 'whitespace only'),
    (None, '', 'None input'),
    (float('nan'), '', 'NaN input'),
    ('😂😂 🔥', '', 'purely symbolic input'),
    ('  multiple   spaces\nand\tnewlines ', 'multiple spaces and newlines', 'whitespace collapse'),
]

failures = []
for raw, expected, desc in cases:
    got = clean_political_text(raw)
    if got != expected:
        failures.append((desc, raw, expected, got))

# Truncation keeps exactly max_length words
long_text = ' '.join(f'w{i}' for i in range(1000))
assert len(clean_political_text(long_text).split()) == 512
assert len(clean_political_text(long_text, max_length=10).split()) == 10
assert clean_political_text('short text', max_length=10) == 'short text'

# Stage-level checks
assert split_hashtags('#StopTheSteal') == 'Stop The Steal'
assert mask_entities('@CNN at https://cnn.com') == '@USER at [URL]'
assert normalize_punctuation('No!!!') == 'No !'
assert remove_emojis('a😀b').split() == ['a', 'b']
assert normalize_unicode('Тrump') == 'Trump'

# Idempotence: cleaning cleaned text changes nothing
for raw, _, _ in cases:
    once = clean_political_text(raw)
    assert clean_political_text(once) == once, f'Not idempotent: {raw!r}'

for desc, raw, expected, got in failures:
    print(f'FAIL [{desc}] {raw!r}: expected {expected!r}, got {got!r}')
assert not failures, f'{len(failures)} assertion(s) failed'
print(f'All {len(cases)} pipeline cases + truncation, stage-level and idempotence checks passed (0 failures).')

All 31 pipeline cases + truncation, stage-level and idempotence checks passed (0 failures).


## Part 4: Before/After Inspection on the Real Corpus
We run the pipeline on the full raw corpus once to see which artifacts it removes and to confirm no record becomes empty. (Phase 4 persists these results into the splits.)

In [5]:
preview_df = raw_df.copy()
preview_df['text_clean'] = preview_df['text'].apply(clean_political_text)

artifacts = {
    'URLs': r'(?:https?://|www\.)\S+',
    '@mentions (raw handles)': r'(?<!\w)@(?!USER\b)\w+',
    'hashtags (#)': r'#\w+',
    'HTML entities': r'&(?:amp|lt|gt|quot|#\d+);',
    'emoji': EMOJI_PATTERN.pattern,
    'repeated !/?': r'[!?]{2,}',
    'repeated dots / …': r'\.{2,}|…',
}
artifact_report = pd.DataFrame({
    'records_before': {k: preview_df['text'].str.contains(p, regex=True).sum() for k, p in artifacts.items()},
    'records_after': {k: preview_df['text_clean'].str.contains(p, regex=True).sum() for k, p in artifacts.items()},
})
display(artifact_report)

empty_after = (preview_df['text_clean'] == '').sum()
print(f'Records empty after cleaning: {empty_after}')
print(f'Mean words  raw: {preview_df["text"].str.split().str.len().mean():.1f} | '
      f'clean: {preview_df["text_clean"].str.split().str.len().mean():.1f}')

changed = preview_df[preview_df['text'] != preview_df['text_clean']]
print(f'Records changed by cleaning: {len(changed):,} / {len(preview_df):,}')
display(changed.sample(8, random_state=42)[['source', 'label', 'text', 'text_clean']])

,records_before,records_after
URLs,4210,0
@mentions (raw handles),976,0
hashtags (#),4615,0
HTML entities,323,0
emoji,64,0
repeated !/?,64,0
repeated dots / …,261,0


Records empty after cleaning: 0
Mean words  raw: 15.2 | clean: 15.8
Records changed by cleaning: 6,093 / 7,481


,source,label,text,text_clean
4542,pheme,0,The most eloquent responses to today's tragedy have come from the cartoonists. #JeSuisCharlie RT @MaximeHaes\n http:...,The most eloquent responses to today's tragedy have come from the cartoonists. Je Suis Charlie RT @USER [URL]
3155,pheme,0,The plot thickens: #Gurlitt's cousin claims Munich #art trove as inheritance http://t.co/Z8k55lzPLp via @YahooNews,The plot thickens: Gurlitt's cousin claims Munich art trove as inheritance [URL] via @USER
4605,pheme,0,"If you're a media outlet reporting on #CharlieHebdo and blurring out the cartoons you're reporting on, you're doing ...","If you're a media outlet reporting on Charlie Hebdo and blurring out the cartoons you're reporting on, you're doing ..."
1531,pheme,0,RT @YourAnonNews: SWAT police just entered a McDonald's in #Ferguson and started asking journalists for ID http://t....,RT @USER: SWAT police just entered a McDonald's in Ferguson and started asking journalists for ID [URL] (via.
3750,pheme,0,NEWS Germanwings flight crashes in south of France http://t.co/Iy99ucFqQ0,NEWS Germanwings flight crashes in south of France [URL]
4285,pheme,1,Gunman at Paris grocery store threatens to kill hostages if police storm the #CharlieHebdo brothers. http://t.co/EBb...,Gunman at Paris grocery store threatens to kill hostages if police storm the Charlie Hebdo brothers. [URL]
1233,pheme,1,"the police version is also that #MikeBrown approached the officer. But if he had JUST robbed a store, why approach p...","the police version is also that Mike Brown approached the officer. But if he had JUST robbed a store, why approach p..."
2014,pheme,0,"The courage it takes to to write these words down, much less display them to the world #Ferguson #MikeBrown http://t...","The courage it takes to to write these words down, much less display them to the world Ferguson Mike Brown [URL]"


## Part 5: Throughput Benchmark
The plan requires `clean_political_text` to process a 1,000-text sample in under 2 seconds, so it stays cheap enough for the FastAPI request path.

In [6]:
sample_texts = raw_df['text'].sample(1000, random_state=42).tolist()

timings = []
for _ in range(5):
    start = time.perf_counter()
    _ = [clean_political_text(t) for t in sample_texts]
    timings.append(time.perf_counter() - start)

best, mean = min(timings), sum(timings) / len(timings)
print(f'1,000 texts: best {best * 1000:.1f} ms | mean {mean * 1000:.1f} ms '
      f'| {1000 / mean:,.0f} texts/sec')
assert mean < 2.0, f'Benchmark too slow: {mean:.2f}s for 1,000 texts'
print('Throughput requirement met (< 2 s per 1,000 texts).')

1,000 texts: best 16.1 ms | mean 16.2 ms | 61,778 texts/sec
Throughput requirement met (< 2 s per 1,000 texts).


## Part 6: Batch Cleaning & Deduplication
Every record is cleaned once into `text_clean`, then deduplicated on lower-cased `text_clean` across both sources **before** any split, so no text can appear in both train and test.

- If every copy in a duplicate group has the same label, the first copy is kept.
- If the copies disagree on the label, every copy is dropped, because the true label cannot be known.

In [7]:
corpus_df = raw_df.rename(columns={'text': 'text_raw'})
corpus_df['text_clean'] = corpus_df['text_raw'].apply(clean_political_text)
corpus_df['word_count'] = corpus_df['text_clean'].str.split().str.len()
assert corpus_df['text_clean'].notna().all() and (corpus_df['text_clean'] != '').all(), 'empty text_clean'

dedup_key = corpus_df['text_clean'].str.lower()
conflicting = corpus_df.groupby(dedup_key)['label'].transform('nunique') > 1
exact_dup = ~conflicting & dedup_key.duplicated()
dedup_df = corpus_df[~conflicting & ~exact_dup].reset_index(drop=True)

EXPECTED_ROWS = 7256
dedup_audit = {
    'Rows before dedup': len(corpus_df),
    'Exact duplicates removed (same label)': int(exact_dup.sum()),
    'Conflicting-label groups dropped': int(dedup_key[conflicting].nunique()),
    'Rows in conflicting-label groups': int(conflicting.sum()),
    'Rows after dedup': len(dedup_df),
    'PolitiFact rows after dedup': int((dedup_df['source'] == 'politifact').sum()),
    'PHEME rows after dedup': int((dedup_df['source'] == 'pheme').sum()),
}
for k, v in dedup_audit.items():
    print(f'{k:<40} {v:>6,}')

assert not dedup_df['text_clean'].str.lower().duplicated().any(), 'duplicates remain'
assert len(dedup_df) == EXPECTED_ROWS, f'Expected {EXPECTED_ROWS:,} rows after dedup, found {len(dedup_df):,}'

Rows before dedup                         7,481
Exact duplicates removed (same label)       203
Conflicting-label groups dropped              9
Rows in conflicting-label groups             22
Rows after dedup                          7,256
PolitiFact rows after dedup                 981
PHEME rows after dedup                    6,275


## Part 7: Stratified 70/15/15 Split on Label × Source
The stratify key is `label_source`, so PolitiFact (about 14% of rows) is spread evenly across the splits. A two-stage `train_test_split` with `random_state=42` takes 70% for train, then halves the remaining 30% into val and test on the same key.

The positive-rate check allows 1 point plus one row's worth of rate (`1/n`), because a small split (for example about 147 PolitiFact val rows) cannot hit a rate closer than one row.

In [8]:
from sklearn.model_selection import train_test_split

strat_key = dedup_df['label'].astype(str) + '_' + dedup_df['source']
train_df, holdout_df = train_test_split(dedup_df, test_size=0.30, stratify=strat_key, random_state=42)
val_df, test_df = train_test_split(holdout_df, test_size=0.50, stratify=strat_key.loc[holdout_df.index],
                                   random_state=42)
splits = {'train': train_df, 'val': val_df, 'test': test_df}

assert sum(len(df) for df in splits.values()) == len(dedup_df)
all_ids = pd.concat([df['id'] for df in splits.values()])
assert all_ids.is_unique, 'id overlap across splits'

rate_rows = []
for name, df in splits.items():
    row = {'split': name, 'rows': len(df), 'overall': df['label'].mean()}
    assert abs(row['overall'] - dedup_df['label'].mean()) < 0.01 + 1 / len(df), (name, 'overall')
    for src, g in dedup_df.groupby('source'):
        part = df[df['source'] == src]
        row[src] = part['label'].mean()
        assert abs(row[src] - g['label'].mean()) < 0.01 + 1 / len(part), (name, src)
    rate_rows.append(row)
rate_rows.append({'split': 'all', 'rows': len(dedup_df), 'overall': dedup_df['label'].mean(),
                  **dedup_df.groupby('source')['label'].mean().to_dict()})
display(pd.DataFrame(rate_rows).set_index('split').round(3))

,rows,overall,pheme,politifact
split,,,,
train,5079,0.382,0.373,0.434
val,1088,0.381,0.373,0.435
test,1089,0.382,0.374,0.435
all,7256,0.382,0.373,0.434


## Part 8: CSV Persistence
Schema: `[id, text_raw, text_clean, label, source, event, word_count]`. `event` is the PHEME event name, or `politifact` for PolitiFact rows, so downstream notebooks can report held-out-event scores. Count features (hashtags, URLs, mentions, `!`, emoji) are not exported: Part 10 shows they add nothing to TF-IDF.

In [9]:
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_COLUMNS = ['id', 'text_raw', 'text_clean', 'label', 'source', 'event', 'word_count']

for name, df in splits.items():
    df[SPLIT_COLUMNS].to_csv(PROCESSED_DIR / f'{name}.csv', index=False)
    print(f'{name + ".csv":<10} {len(df):>5,} rows')

train.csv  5,079 rows
val.csv    1,088 rows
test.csv   1,089 rows


## Part 9: PHEME Cascade Features
One row per cascade (all 6,425, including cascades whose source text was removed by dedup). `split` ties each cascade to its text row so the spread regressor in notebook 07 uses the same partition.

- `max_depth` is the longest BFS distance from the `thread_id` node, which does not assume a clean tree (about 20% of graphs are not). Nodes that no path from `thread_id` reaches are left out of the depth.
- There is no `propagation_duration`: the graphs carry no timestamps.
- `pagerank`, `degree_centrality` and in/out degree are not exported. They are computed over the whole cascade, so they leak the size target.

In [10]:
import networkx as nx

split_of = {cid: name for name, df in splits.items()
            for cid in df.loc[df['source'] == 'pheme', 'id']}

cascade_rows = []
for g in cascades:
    thread_id = str(g.graph['thread_id'])
    root = g.nodes[thread_id]
    cascade_rows.append({
        'cascade_id': thread_id,
        'label': int(g.graph['label']),
        'event': str(g.graph['event']),
        'split': split_of.get(thread_id, 'excluded_dup'),
        'cascade_size': g.number_of_nodes(),
        'max_depth': max(nx.single_source_shortest_path_length(g, thread_id).values()),
        'reply_count': g.out_degree(thread_id),
        'root_user_followers': root['user_followers'],
        'root_user_verified': int(root['user_verified']),
        'n_unique_verified': sum(1 for _, v in g.nodes(data='user_verified') if v == 1),
    })
cascade_df = pd.DataFrame(cascade_rows)

assert len(cascade_df) == 6425, f'Expected 6,425 cascades, found {len(cascade_df):,}'
assert cascade_df.notna().all().all(), 'null cascade features'
assert (cascade_df['max_depth'] >= 0).all()
for name in splits:
    assert set(cascade_df.loc[cascade_df['split'] == name, 'cascade_id']) == \
        {k for k, v in split_of.items() if v == name}, f'cascade split mismatch: {name}'

cascade_df.to_csv(PROCESSED_DIR / 'cascade_features.csv', index=False)
print(cascade_df['split'].value_counts().to_string())
display(cascade_df.describe().round(2))

split
train           4392
test             942
val              941
excluded_dup     150


,label,cascade_size,max_depth,reply_count,root_user_followers,root_user_verified,n_unique_verified
count,6425.00,6425.00,6425.00,6425.00,6425.00,6425.00,6425.00
mean,0.37,16.46,3.21,8.48,1160176.75,0.52,0.73
std,0.48,19.79,3.80,6.44,3173751.59,0.50,1.26
min,0.00,1.00,0.00,0.00,4.00,0.00,0.00
25%,0.00,5.00,1.00,3.00,7620.00,0.00,0.00
50%,0.00,12.00,2.00,8.00,56488.00,1.00,1.00
75%,1.00,21.00,4.00,13.00,542978.00,1.00,1.00
max,1.00,346.00,47.00,123.00,25303087.00,1.00,37.00


## Part 10: Event-Shortcut Baseline
Logistic regression on three feature sets, each scored by Macro F1 two ways:

- **Random split:** fit on `train`, score on `test` (the split from Part 7).
- **Events it never saw:** leave one PHEME event out. Fit on every other PHEME event plus the PolitiFact train rows, score on the held-out event, and average over events with at least 100 threads.

Fixed setup: word 1–2-gram TF-IDF (`min_df=2`, `sublinear_tf=True`) on `text_clean`; count features (hashtags, URLs, mentions, `!`, emoji) counted on `text_raw`, because cleaning removes them, and standardised; `LogisticRegression(class_weight='balanced', max_iter=3000)`.

In [11]:
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

COUNT_PATTERNS = [r'#\w+', URL_PATTERN.pattern, MENTION_PATTERN.pattern, '!', EMOJI_PATTERN.pattern]
FEATURE_SETS = {
    'Count features only (hashtags, URLs, mentions, !, emoji)': (False, True),
    'TF-IDF on cleaned text': (True, False),
    'TF-IDF + count features': (True, True),
}
MIN_EVENT_THREADS = 100


def count_features(texts: pd.Series) -> np.ndarray:
    return np.column_stack([texts.str.count(p, flags=re.IGNORECASE if p == URL_PATTERN.pattern else 0)
                            for p in COUNT_PATTERNS]).astype(float)


def macro_f1(fit_df: pd.DataFrame, eval_df: pd.DataFrame, use_tfidf: bool, use_counts: bool) -> float:
    fit_parts, eval_parts = [], []
    if use_tfidf:
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
        fit_parts.append(vec.fit_transform(fit_df['text_clean']))
        eval_parts.append(vec.transform(eval_df['text_clean']))
    if use_counts:
        scaler = StandardScaler()
        fit_parts.append(csr_matrix(scaler.fit_transform(count_features(fit_df['text_raw']))))
        eval_parts.append(csr_matrix(scaler.transform(count_features(eval_df['text_raw']))))
    model = LogisticRegression(class_weight='balanced', max_iter=3000)
    model.fit(hstack(fit_parts).tocsr(), fit_df['label'])
    return f1_score(eval_df['label'], model.predict(hstack(eval_parts).tocsr()), average='macro')


pheme_df_dedup = dedup_df[dedup_df['source'] == 'pheme']
event_sizes = pheme_df_dedup['event'].value_counts()
held_out_events = event_sizes[event_sizes >= MIN_EVENT_THREADS].index.tolist()
politifact_train = train_df[train_df['source'] == 'politifact']

summary_rows, per_event_rows = [], []
for name, (use_tfidf, use_counts) in FEATURE_SETS.items():
    random_f1 = macro_f1(train_df, test_df, use_tfidf, use_counts)
    event_f1 = {}
    for event in held_out_events:
        fit_df = pd.concat([pheme_df_dedup[pheme_df_dedup['event'] != event], politifact_train])
        event_f1[event] = macro_f1(fit_df, pheme_df_dedup[pheme_df_dedup['event'] == event], use_tfidf, use_counts)
    summary_rows.append({'Model': name, 'Random split': random_f1, 'Events it never saw': np.mean(list(event_f1.values()))})
    per_event_rows += [{'Model': name, 'Event': e, 'Macro F1': f} for e, f in event_f1.items()]

baseline_df = pd.DataFrame(summary_rows)
per_event_df = pd.DataFrame(per_event_rows).pivot(index='Event', columns='Model', values='Macro F1')[list(FEATURE_SETS)]
display(baseline_df.round(3))
display(per_event_df.round(3))

,Model,Random split,Events it never saw
0,"Count features only (hashtags, URLs, mentions, !, emoji)",0.544,0.475
1,TF-IDF on cleaned text,0.843,0.597
2,TF-IDF + count features,0.843,0.598


Model,"Count features only (hashtags, URLs, mentions, !, emoji)",TF-IDF on cleaned text,TF-IDF + count features
Event,,,
charliehebdo-all-rnr-threads,0.436,0.749,0.752
ferguson-all-rnr-threads,0.450,0.590,0.592
germanwings-crash-all-rnr-threads,0.470,0.738,0.744
gurlitt-all-rnr-threads,0.480,0.566,0.591
ottawashooting-all-rnr-threads,0.430,0.716,0.711
prince-toronto-all-rnr-threads,0.376,0.195,0.173
putinmissing-all-rnr-threads,0.662,0.504,0.516
sydneysiege-all-rnr-threads,0.499,0.717,0.703


### Write `backend/reports/split_baselines.md`
The report holds the dedup audit, split sizes, both baseline tables and per-event label counts. Some events are almost entirely one label (`prince-toronto`), so their Macro F1 is unstable and must be read next to the label counts.

In [12]:
def md_table(df: pd.DataFrame, floatfmt: str = '.3f') -> str:
    header = '| ' + ' | '.join(map(str, df.columns)) + ' |'
    sep = '|' + '|'.join('---' for _ in df.columns) + '|'
    fmt = lambda v: format(v, floatfmt) if isinstance(v, float) else (f'{v:,}' if isinstance(v, (int, np.integer)) else str(v))
    return '\n'.join([header, sep] + ['| ' + ' | '.join(fmt(v) for v in row) + ' |' for row in df.itertuples(index=False)])


event_labels = pheme_df_dedup.groupby('event')['label'].agg(rumour='sum', threads='size')
event_labels['non_rumour'] = event_labels['threads'] - event_labels['rumour']
event_table = per_event_df.join(event_labels).reset_index().rename(columns={'threads': 'Threads', 'rumour': 'Rumour', 'non_rumour': 'Non-rumour'})
event_table = event_table[['Event', 'Threads', 'Rumour', 'Non-rumour'] + list(FEATURE_SETS)]
event_table.columns = ['Event', 'Threads', 'Rumour', 'Non-rumour', 'Count only', 'TF-IDF', 'TF-IDF + count']
split_table = pd.DataFrame(rate_rows).rename(columns={'split': 'Split', 'rows': 'Rows', 'overall': 'Positive rate',
                                                      'pheme': 'PHEME positive rate',
                                                      'politifact': 'PolitiFact positive rate'})
best = baseline_df.loc[baseline_df['Model'] == 'TF-IDF on cleaned text'].iloc[0]
partial_reach = sum(1 for g in cascades
                    if len(nx.single_source_shortest_path_length(g, str(g.graph['thread_id']))) < g.number_of_nodes())

report = f"""# Split Baselines and Dedup Audit

Generated by `backend/notebooks/02_preprocessing.ipynb` (Parts 6-10). Re-run the notebook to refresh it.

## Dedup audit

Dedup key is lower-cased `text_clean` across both sources, applied before splitting. Groups whose copies
disagree on the label are dropped entirely.

{md_table(pd.DataFrame(list(dedup_audit.items()), columns=['Step', 'Rows']))}

## Splits

Stratified on `label` x `source`, `random_state=42`.

{md_table(split_table)}

`cascade_features.csv` holds all 6,425 PHEME cascades; {int((cascade_df['split'] == 'excluded_dup').sum())} are marked
`excluded_dup` because dedup removed their source text. In {partial_reach} cascades some nodes are not reachable from
the `thread_id` node, so `max_depth` covers only the reachable part.

## Baseline Macro F1: random split vs events the model never saw

{md_table(baseline_df)}

"Events it never saw" is the mean over {len(held_out_events)} PHEME events with at least {MIN_EVENT_THREADS} threads, each held out in turn
(fit on the other events plus PolitiFact train rows).

Setup: word 1-2-gram TF-IDF (`min_df=2`, `sublinear_tf=True`) on `text_clean`; count features on `text_raw`,
standardised; `LogisticRegression(class_weight='balanced', max_iter=3000)`.

### Finding

TF-IDF scores {best['Random split']:.3f} Macro F1 on the random split but {best['Events it never saw']:.3f} on events it never saw.
A simple model partly learns the news event rather than misinformation itself. Notebooks 04 to 07 must report
both numbers, which is why `event` stays in every split CSV. Count features add nothing to TF-IDF.

The exploratory figures in the plan (0.927 random, 0.735 unseen events) were not reproduced: their setup was not
recorded, and the TF-IDF configurations tried here (word and character n-grams, C of 1 and 10, with and without class
weights, with and without dedup) scored 0.81-0.86 on the random split and 0.52-0.60 on unseen events. The gap
between the two settings, which is the finding, is larger than first reported.

## Per-event Macro F1 (held-out event)

{md_table(event_table)}

Events that are almost entirely one label (for example `prince-toronto`) give unstable Macro F1: a model that
predicts the majority label still scores below 0.5 there.

## Label semantics

PHEME labels are **rumour vs non-rumour**; PolitiFact labels are **fake vs real**. `label=1` means rumour or fake.
A PHEME rumour is not necessarily false. Many PolitiFact "real" titles are page names rather than claims
(for example "Transcripts"); dedup removes their repeats, but single copies remain as a known dataset limitation.
"""

REPORT_PATH = Path('../reports/split_baselines.md')
REPORT_PATH.write_text(report)
print(f'Wrote {REPORT_PATH} ({len(report.splitlines())} lines)')

Wrote ../reports/split_baselines.md (80 lines)
